In [117]:
import pandas as pd

In [118]:
users = pd.read_csv("../data/raw/users_data.csv")
print(users.head())

  user_id first_name last_name  age      gender   birthdate sign_up_date  \
0  user_1      Chris    Wilson   56      Female  2000-02-29   2023-02-06   
1  user_2    Michael    Miller   46  Non-binary  1978-12-22   2023-08-08   
2  user_3     Daniel     Smith   32      Female  1962-08-30   2021-01-11   
3  user_4      David     Smith   60        Male  2003-12-05   2023-08-07   
4  user_5      Chris     Jones   25      Female  2004-08-25   2021-01-08   

  user_location subscription_plan  
0        Denver             Basic  
1       Orlando               Pro  
2       Orlando             Basic  
3        Denver               Pro  
4        Denver             Basic  


In [119]:
print("Dataset size:", users.shape); print("Missing values:\n", users.isnull().sum()); print("Duplicate rows:", users.duplicated().sum()); print("Data types:\n", users.dtypes)

Dataset size: (5000, 9)
Missing values:
 user_id              0
first_name           0
last_name            0
age                  0
gender               0
birthdate            0
sign_up_date         0
user_location        0
subscription_plan    0
dtype: int64
Duplicate rows: 0
Data types:
 user_id                str
first_name             str
last_name              str
age                  int64
gender                 str
birthdate              str
sign_up_date           str
user_location          str
subscription_plan      str
dtype: object


In [120]:
u = users.copy(); u["birthdate"] = pd.to_datetime(u["birthdate"], errors="coerce"); u["sign_up_date"] = pd.to_datetime(u["sign_up_date"], errors="coerce"); u["calculated_age"] = u["sign_up_date"].dt.year - u["birthdate"].dt.year - ((u["sign_up_date"].dt.month < u["birthdate"].dt.month) | ((u["sign_up_date"].dt.month == u["birthdate"].dt.month) & (u["sign_up_date"].dt.day < u["birthdate"].dt.day))); print("Invalid dates:", u[["birthdate", "sign_up_date"]].isna().sum().sum()); print("Age mismatches:", (u["age"] != u["calculated_age"]).sum()); display(u.loc[u["age"] != u["calculated_age"], ["user_id", "age", "birthdate", "sign_up_date", "calculated_age"]].head())

Invalid dates: 0
Age mismatches: 4907


,user_id,age,birthdate,sign_up_date,calculated_age
0,user_1,56,2000-02-29,2023-02-06,22
1,user_2,46,1978-12-22,2023-08-08,44
2,user_3,32,1962-08-30,2021-01-11,58
3,user_4,60,2003-12-05,2023-08-07,19
4,user_5,25,2004-08-25,2021-01-08,16


In [121]:
print("Recorded age range:", u["age"].min(), "-", u["age"].max()); print("Calculated age range:", u["calculated_age"].min(), "-", u["calculated_age"].max()); print("Recorded ages outside 15-100:", ((u["age"] < 15) | (u["age"] > 100)).sum()); print("Calculated ages outside 15-100:", ((u["calculated_age"] < 15) | (u["calculated_age"] > 100)).sum())

Recorded age range: 18 - 64
Calculated age range: 15 - 65
Recorded ages outside 15-100: 0
Calculated ages outside 15-100: 0


In [122]:
attendance = pd.read_csv("../data/raw/checkin_checkout_history_updated.csv"); print("Dataset size:", attendance.shape); print("Columns:", attendance.columns.tolist()); print("Missing values:\n", attendance.isnull().sum()); print("Duplicate rows:", attendance.duplicated().sum()); display(attendance.head(3))

Dataset size: (300000, 6)
Columns: ['user_id', 'gym_id', 'checkin_time', 'checkout_time', 'workout_type', 'calories_burned']
Missing values:
 user_id            0
gym_id             0
checkin_time       0
checkout_time      0
workout_type       0
calories_burned    0
dtype: int64
Duplicate rows: 0


,user_id,gym_id,checkin_time,checkout_time,workout_type,calories_burned
0,user_3291,gym_6,2023-09-10 15:55:00,2023-09-10 16:34:00,Weightlifting,462
1,user_1944,gym_2,2023-04-13 20:07:00,2023-04-13 22:43:00,Yoga,1278
2,user_958,gym_7,2023-06-10 12:24:00,2023-06-10 13:49:00,Cardio,858


In [123]:
a = attendance.copy(); a["checkin_time"] = pd.to_datetime(a["checkin_time"], errors="coerce"); a["checkout_time"] = pd.to_datetime(a["checkout_time"], errors="coerce"); a["duration_minutes"] = (a["checkout_time"] - a["checkin_time"]).dt.total_seconds() / 60; print("Invalid timestamps:", a[["checkin_time", "checkout_time"]].isna().sum().sum()); print("Checkout before check-in:", (a["duration_minutes"] < 0).sum()); print("Zero-minute sessions:", (a["duration_minutes"] == 0).sum()); print(a["duration_minutes"].describe())

Invalid timestamps: 0
Checkout before check-in: 0
Zero-minute sessions: 0
count    300000.000000
mean        104.519627
std          43.216521
min          30.000000
25%          67.000000
50%         105.000000
75%         142.000000
max         179.000000
Name: duration_minutes, dtype: float64


In [124]:
locations = pd.read_csv("../data/raw/gym_locations_data.csv"); merged = a.merge(u[["user_id", "sign_up_date"]], on="user_id", how="left", indicator=True); print("Attendance with unknown users:", (merged["_merge"] == "left_only").sum()); print("Attendance before signup:", (merged["checkin_time"] < merged["sign_up_date"]).sum()); print("Attendance with unknown gyms:", (~a["gym_id"].isin(locations["gym_id"])).sum())

Attendance with unknown users: 0
Attendance before signup: 42932
Attendance with unknown gyms: 0


In [125]:
print("Affected attendance records:", (merged["checkin_time"] < merged["sign_up_date"]).sum()); print("Affected unique users:", merged.loc[merged["checkin_time"] < merged["sign_up_date"], "user_id"].nunique()); display(merged.loc[merged["checkin_time"] < merged["sign_up_date"], ["user_id", "checkin_time", "sign_up_date"]].head(10))

Affected attendance records: 42932
Affected unique users: 1416


,user_id,checkin_time,sign_up_date
1,user_1944,2023-04-13 20:07:00,2023-07-16
13,user_4489,2023-04-06 17:32:00,2023-09-27
17,user_4922,2023-05-29 16:01:00,2023-07-04
26,user_2242,2023-01-16 07:34:00,2023-03-09
30,user_2853,2023-05-01 18:52:00,2023-05-19
34,user_2879,2023-04-04 09:23:00,2023-09-09
37,user_1807,2023-03-25 12:11:00,2023-06-20
39,user_3503,2023-04-05 14:36:00,2023-04-12
57,user_12,2023-07-05 06:22:00,2023-10-02
75,user_3408,2023-03-05 06:56:00,2023-09-13


In [126]:
attendance_flagged = merged[merged["checkin_time"] < merged["sign_up_date"]].copy(); attendance_clean = merged[merged["checkin_time"] >= merged["sign_up_date"]].copy(); print("Records flagged for review:", len(attendance_flagged)); print("Records retained for analysis:", len(attendance_clean)); print("Total records:", len(attendance_flagged) + len(attendance_clean))

Records flagged for review: 42932
Records retained for analysis: 257068
Total records: 300000


In [127]:
plans = pd.read_csv("../data/raw/subscription_plans.csv"); print("=== SUBSCRIPTION PLANS ==="); print("Size:", plans.shape); print("Missing values:\n", plans.isnull().sum()); print("Duplicate rows:", plans.duplicated().sum()); display(plans); print("=== GYM LOCATIONS ==="); print("Size:", locations.shape); print("Missing values:\n", locations.isnull().sum()); print("Duplicate rows:", locations.duplicated().sum()); display(locations)

=== SUBSCRIPTION PLANS ===
Size: (3, 3)
Missing values:
 subscription_plan    0
price_per_month      0
features             0
dtype: int64
Duplicate rows: 0


,subscription_plan,price_per_month,features
0,Basic,19.99,"Access to basic gym facilities, Limited class ..."
1,Pro,49.99,"Access to all facilities, Unlimited class acce..."
2,Student,9.99,"Access to basic facilities, Limited class acce..."


=== GYM LOCATIONS ===
Size: (10, 4)
Missing values:
 gym_id        0
location      0
gym_type      0
facilities    0
dtype: int64
Duplicate rows: 0


,gym_id,location,gym_type,facilities
0,gym_1,New York,Premium,"Climbing Wall, Swimming Pool, Basketball Court"
1,gym_2,Los Angeles,Budget,"Climbing Wall, Yoga Classes, Sauna"
2,gym_3,Chicago,Budget,"Sauna, Climbing Wall, Swimming Pool"
3,gym_4,Houston,Premium,"Climbing Wall, Basketball Court, Swimming Pool"
4,gym_5,Phoenix,Standard,"Basketball Court, CrossFit, Swimming Pool"
5,gym_6,Philadelphia,Budget,"Swimming Pool, Climbing Wall, Sauna"
6,gym_7,San Antonio,Premium,"Sauna, Basketball Court, Swimming Pool"
7,gym_8,San Diego,Standard,"Basketball Court, Sauna, CrossFit"
8,gym_9,Dallas,Premium,"Sauna, CrossFit, Yoga Classes"
9,gym_10,San Jose,Premium,"Swimming Pool, Sauna, CrossFit"


In [128]:
print("Duplicate user IDs:", users["user_id"].duplicated().sum()); print("Duplicate gym IDs:", locations["gym_id"].duplicated().sum()); print("Duplicate plan names:", plans["subscription_plan"].duplicated().sum()); print("Members with unknown plans:", (~users["subscription_plan"].isin(plans["subscription_plan"])).sum())

Duplicate user IDs: 0
Duplicate gym IDs: 0
Duplicate plan names: 0
Members with unknown plans: 0


In [129]:
users.to_csv("../data/processed/users.csv", index=False)
attendance_clean.to_csv("../data/processed/attendance.csv", index=False)
locations.to_csv("../data/processed/locations.csv", index=False)
plans.to_csv("../data/processed/plans.csv", index=False)

print("Saved successfully!")

Saved successfully!
